In [5]:
%load_ext autoreload
%autoreload 2
%load_ext line_profiler

In [6]:
from tqdm import tqdm

import os
import torch
import numpy as np
from torch_geometric.data import Batch, HeteroData
from scipy.sparse import coo_array
from numpy.linalg import LinAlgError

import gurobipy as gp
from gurobipy import GRB

In [7]:
rng = np.random.RandomState(1)

In [1]:
root = 'datasets/maxcut_er_50_0.1'
os.mkdir(root)
os.mkdir(os.path.join(root, 'processed'))

NameError: name 'os' is not defined

### Generic

In [2]:
# density = 0.01
# nrows = 500
# ncols = 500

# def surrogate_gen():
#     assert max(nrows, ncols) * density > 1

#     m, n = min(nrows, ncols), max(nrows, ncols)

#     # make sure rows and cols are selected at least once
#     rows = np.hstack([np.arange(m), np.random.randint(0, m, (n - m,))])
#     cols = np.arange(n)

#     # generate the rest
#     nnz = int(nrows * ncols * density)
#     num_rest = nnz - n

#     rows_rest = np.random.randint(0, m, (num_rest,))
#     cols_rest = np.random.randint(0, n, (num_rest,))

#     values = np.random.randn(nnz)

#     A = coo_array((values, (np.hstack([rows, rows_rest]), np.hstack([cols, cols_rest]))), shape=(m, n)).toarray()

#     x_feas = np.abs(np.random.randn(ncols))  # Ensure x_feas is non-negative
#     b = A @ x_feas + np.abs(np.random.randn(nrows))  # Ensure feasibility

#     c = np.abs(np.random.randn(ncols))
#     return A, b, c

# bounds = None

### Max cut

In [3]:
import cvxpy as cp
from torch_geometric.utils import to_dense_adj
from generate_sdp_instances import erdos_renyi_generator

def generate_max_cut_sdp(nnodes, density):
    data = erdos_renyi_generator(rng, nnodes, nnodes, density, density)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]
    adj = to_dense_adj(edge_index, max_num_nodes=N)[0].numpy()

    A = []
    b = []
    # diagonals being 1
    for i in range(N):
        const = np.zeros((N, N))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    return adj, np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)


def solve_sdp_cvxpy(C, A, b, norm_strength=0.):
    N = C.shape[0]
    # Define and solve the CVXPY problem.
    # Create a symmetric matrix variable.
    X = cp.Variable((N, N), PSD=True)

    # The operator >> denotes matrix inequality.
    # constraints = [X >> 0]
    constraints = [cp.trace(A[..., i] @ X) == b[i] for i in range(N)]
    objective = cp.trace(C @ X)
    # wrt the min norm
    if norm_strength > 0:
        objective += cp.sum_squares(X) * norm_strength
    prob = cp.Problem(cp.Minimize(objective), constraints)
    prob.solve(verbose=False, solver=cp.MOSEK)

    # Print result.
    sol = prob.value
    X = X.value
    (eigenval, eigenvec) = np.linalg.eig(X)
    #print('Eigenvalues--Note how sparse they are:', eigenval)

    # ensure eigenvalues are positive
    # pad by .001 for precision issues with cholesky decomposition
    if np.min(eigenval) <= 0:
        X = X + (1.e-5 - np.min(eigenval)) * np.eye(N)
    V = np.linalg.cholesky(X)
    #print('Cholesky Decomposition:', V)

    return sol, X, V, prob.status, prob.solver_stats.solve_time

(CVXPY) Jun 22 08:10:07 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.13.4784). Expected < 9.12.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jun 22 08:10:07 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.13.4784). Expected < 9.12.0. Please open a feature request on cvxpy to enable support for this version.')


In [8]:
A = np.array([[[1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1.]]], dtype=np.float32)

b = np.array([1., 1., 1., 1., 1.], dtype=np.float32)

C = np.array([[0., 1., 0., 0., 0.],
               [1., 0., 1., 1., 1.],
               [0., 1., 0., 1., 0.],
               [0., 1., 1., 0., 0.],
               [0., 1., 0., 0., 0.]], dtype=np.float32)

In [9]:
sol, X, V, stat, times = solve_sdp_cvxpy(C, A, b, 1.e-5)

In [10]:
X.round(4)

array([[ 1. , -1. ,  0.5,  0.5,  1. ],
       [-1. ,  1. , -0.5, -0.5, -1. ],
       [ 0.5, -0.5,  1. , -0.5,  0.5],
       [ 0.5, -0.5, -0.5,  1. ,  0.5],
       [ 1. , -1. ,  0.5,  0.5,  1. ]])

In [11]:
np.trace(C @ X)

np.float64(-6.999999983179281)

In [12]:
E, U = np.linalg.eigh(C)

In [13]:
E

array([-1.8136065e+00, -1.0000000e+00, -2.0595711e-16,  4.7068343e-01,
        2.3429232e+00], dtype=float32)

In [14]:
U[2]

array([-2.6055464e-01, -7.0710677e-01,  4.0442906e-16,  4.5598480e-01,
       -4.7348616e-01], dtype=float32)

In [15]:
U[3]

array([-2.6055464e-01,  7.0710677e-01,  3.4891791e-16,  4.5598480e-01,
       -4.7348616e-01], dtype=float32)

# create ineq

In [16]:
from cvxpy import DCPError, DGPError, DPPError, SolverError

In [17]:
nnodes = 50
dens = 0.1
fnorm_strength = 1.e-3
eigs = 50

In [18]:
graphs = []
pkg_idx = 0
success_cnt = 0

max_iter = 12000
num = 10000

pbar = tqdm(range(max_iter))
for i in pbar:
    C, A, b = generate_max_cut_sdp(nnodes, dens)
    try:
        assert np.any(C != 0.)
        eigval, eigvec = np.linalg.eigh(C)
        eigvec *= (np.abs(eigval) ** 0.5)[None]
        eigval = np.sign(eigval)
        sol, X, V, stat, times = solve_sdp_cvxpy(C, A, b, fnorm_strength)
        assert stat == 'optimal'
    except (LinAlgError, DCPError, DGPError, DPPError, SolverError, AssertionError):
        continue

    else:
        m = b.shape[0]
        n = C.shape[0]
        A = torch.from_numpy(A).float()
        A = A.reshape(-1, A.shape[-1]).T  # m, n**2
        A_where = torch.where(A)
        
        c2v_idx = torch.vstack(A_where)
        c2v_value = A[A_where][:, None]
        
        eigvec = torch.from_numpy(eigvec).float()[:, -eigs:]  # n, num_eig
        eigval = torch.from_numpy(eigval).float()[-eigs:]  # num_eig
        C = torch.from_numpy(C).float().reshape(-1)[None]
        # sparse vals obj connections
        C_where = torch.where(C)
        o2v_idx = torch.vstack(C_where)
        o2v_value = C[C_where][:, None]

        x = torch.from_numpy(X).float().reshape(-1)

        data = HeteroData(
            cons={
                'num_nodes': m,
                'x': torch.empty(m, 0),
                 },
            vals={
                'num_nodes': n ** 2,
                'x': torch.empty(n ** 2, 0),
            },
            obj={
                'num_nodes': 1,
                'x': torch.ones(1).float(),
                 },
            cons__to__vals={'edge_index': c2v_idx,
                            'edge_attr': c2v_value},
            obj__to__vals={'edge_index': o2v_idx,
                            'edge_attr': o2v_value},
            x_solution=x,
            obj_solution=torch.tensor([sol]),
            b=torch.from_numpy(b).float(),
            c_eigval=eigval,
            c_eigvec=eigvec,
        )
        success_cnt += 1
        graphs.append(data)

    if len(graphs) >= 1000 or success_cnt == num:
        torch.save(Batch.from_data_list(graphs), f'{root}/processed/batch{pkg_idx}.pt')
        pkg_idx += 1
        graphs = []

    if success_cnt >= num:
        break

    pbar.set_postfix({'suc': success_cnt})

/var/folders/6v/m172f7bs02l0tkwy4_9yxttm0000gn/T/ipykernel_76240/731973344.py:39: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1720538194616/work/aten/src/ATen/native/Copy.cpp:305.)
  x = torch.from_numpy(X).float().reshape(-1)
 83%|███████████████████████████████████████████████████████████████████████████████████████████████████▏                   | 9999/12000 [47:05<09:25,  3.54it/s, suc=9999]


In [19]:
from data.dataset import LPDataset

In [20]:
ds = LPDataset(root, 'test')

Processing...
/Users/qianchendi/SDP/data/dataset.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data_list.extend(Batch.to_data_list(torch.load(osp.join(self.processed_

In [21]:
data = ds[0]